This notebook can be run in Google Colab by setting `colab = True` in the first cell of **Setup** below. The second cell will then install all relevant libraries to the current Colab session.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lu-liang-geo/Sam-Lidar/blob/main/notebooks/Paper_Methods.ipynb)

# Setup

In [ ]:
# If running in colab, set colab = True, else set colab = False.
colab = True

In [ ]:
%%capture
if colab == True:

    # Clone repositories for DeepForest, Detectree2, SAM 2, and Sam-Lidar
    !git clone 'https://github.com/weecology/DeepForest'
    !git clone 'https://github.com/PatBall1/detectree2.git'
    !git clone 'https://github.com/facebookresearch/detectron2.git'
    !git clone 'https://github.com/facebookresearch/segment-anything-2.git'
    !git clone 'https://github.com/lu-liang-geo/Sam-Lidar.git'

    # Install code from this repository
    !pip install "/content/Sam-Lidar[deepforest,detectree2,sam2]"

    # Download weights for Detectree2 and SAM 2
    !mkdir /content/sam2_weights
    !wget -q -P /content/sam2_weights https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

    !mkdir /content/detectree2_weights
    !wget -q -P /content/detectree2_weights https://zenodo.org/records/12773341/files/230103_randresize_full.pth

    # Download annotated data
    !pip install zenodo_get
    !zenodo_get https://doi.org/10.5281/zenodo.15133613
    !unzip '/content/data.zip' -d '/content'

In [ ]:
# Import libraries
import os
import torch
import shutil
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import supervision as sv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import linear_sum_assignment
from IPython.display import display
from copy import deepcopy

In [ ]:
# Import installed libraries
if colab == True:
    deepforest_path = '/content/DeepForest/src'
    detectron_path = '/content/detectron2'
    detectree_path = '/content/detectree2'
    sam_path = '/content/segment-anything-2'
    sam_lidar_path = '/content/'
else:
    deepforest_path = 'PATH/TO/DeepForest/src'
    detectron_path = 'PATH/TO/detectron2/detectron2'
    detectree_path = 'PATH/TO/detectree2/detectree2'
    sam_path = 'PATH/TO/segment-anyything-2/sam2'
    sam_lidar_path = 'PATH/TO/SAM-Lidar/sam_lidar'

os.chdir(deepforest_path)
from deepforest import main

os.chdir(detectron_path)
from detectron2.engine import DefaultPredictor

os.chdir(detectree_path)
from detectree2.models.train import setup_cfg

os.chdir(sam_path)
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

os.chdir(sam_lidar_path)
from sam_lidar import (
    NEONTreeDataset,
    rasterize_lidar,
    lidar_filter,
    sample_points,
    show_as_points,
    segment_boxes,
    segment_box_lidar,
    custom_nms,
    hungarian_matching,
    per_tree_metrics,
    per_tree_std,
    per_box_metrics
)

In [ ]:
# Set device, either CPU or CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# Load DeepForest Model
deepforest_model = main.deepforest()
deepforest_model.use_release()
deepforest_model.to(device)
print()

In [ ]:
# Load Detectree2 Model
if colab == True:
    detectron2_folder = "/content/detectron2"
    detectree_weights = "/content/detectree2_weights/230103_randresize_full.pth"
else:
    detectron2_folder = "PATH/TO/DETECTRON2/DIRECTORY"
    detectree_weights = "PATH/TO/DETECTREE2/WEIGHTS.pth"

detectron_model = 'mask_rcnn_R_101_FPN_3x.yaml'
detectron_base = 'Base-RCNN-FPN.yaml'

model_path = os.path.join(detectron2_folder, 'configs/COCO-InstanceSegmentation', detectron_model)
base_path = os.path.join(detectron2_folder, 'configs', detectron_base)
model_zoo = os.path.join(detectron2_folder, 'detectron2/model_zoo/configs')

os.makedirs(os.path.join(model_zoo, 'COCO-InstanceSegmentation'))
shutil.copy(model_path, os.path.join(model_zoo, 'COCO-InstanceSegmentation'))
shutil.copy(base_path, model_zoo)

cfg = setup_cfg(update_model=detectree_weights)
cfg.MODEL.DEVICE = device.type
detectree_model=DefaultPredictor(cfg)

In [ ]:
# Load SAM Model. We're currently using SAM-2
if colab == True:
    sam_weights = "/content/sam2_weights/sam2_hiera_large.pt"
else:
    sam_weights = "PATH/TO/SAM2/WEIGHTS.pt"

sam_model = build_sam2("sam2_hiera_l.yaml", sam_weights, device=device)
sam_predictor = SAM2ImagePredictor(sam_model)

In [ ]:
# Define dataset
if colab == True:
    rgb_path = "/content/NeonSubset/data/RGB"
    lidar_path = "/content/NeonSubset/data/LiDAR"
    ann_path = "/content/NeonSubset/data/Annotations"
else:
    rgb_path = "PATH/TO/RGB/FOLDER"
    lidar_path = "PATH/TO/LIDAR/FOLDER"
    ann_path = "PATH/TO/ANNOTATION/FOLDER"

dataset = NEONTreeDataset(rgb_folder=rgb_path, lidar_folder=lidar_path, ann_folder=ann_path)

# Set Image Paths / Random Seed

In [ ]:
# Set random seed for reproducibility. Seed used for paper is 1.
seed = 1

# Define test images and classifications
test_imgs = [('2018_SJER_3_252000_4106000_image_234', 'Low'), ('2018_SJER_3_254000_4108000_image_700', 'Low'),
             ('2018_TEAK_3_318000_4107000_image_718', 'Low'), ('DSNY_002_2018', 'Low'), ('JERC_055_2018', 'Low'),
             ('OSBS_051_2019', 'Low'), ('ABBY_063_2019', 'Medium'), ('BONA_018_2019', 'Medium'), ('DSNY_025_2018', 'Medium'),
             ('SJER_025_2018', 'Medium'), ('TEAK_049_2018', 'Medium'), ('WREF_072_2019', 'Medium'), ('ABBY_076_2019', 'High'),
             ('BART_050_2019', 'High'), ('BONA_005_2019', 'High'), ('DELA_047_2019', 'High'), ('LENO_066_2019', 'High'),
             ('SERC_004_2019', 'High')]

# Define metrics dictionary that will be updated below
metrics = dict()

Print number of trees in each density class.

In [ ]:
trees = {'Low':dict(), 'Medium':dict(), 'High':dict()}

for filename, density in test_imgs:
  img = dataset.get_image(filename)
  true_boxes = img['box']
  trees[density][filename] = len(true_boxes)

total_trees = 0
for density in ['Low','Medium','High']:
  density_trees = 0
  print(density)
  for filename, number in trees[density].items():
    print(filename, ':', number)
    density_trees += number
    total_trees += number
  print()
  print(density, 'Trees :', density_trees)
  print()
print('Total Trees :', total_trees)

Show three images from the dataset.

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4))

for i, (filename, density) in enumerate([('2018_SJER_3_254000_4108000_image_700', 'Low'),
                                            ('WREF_072_2019', 'Medium'), ('DELA_047_2019', 'High')]):
  img = dataset.get_image(filename)
  rgb_img = img['rgb']
  axs[i].imshow(rgb_img)
  axs[i].set_frame_on(False)
  axs[i].get_xaxis().set_ticks([])
  axs[i].get_yaxis().set_ticks([])

plt.tight_layout()
#plt.savefig('Tree Plots.jpg')
plt.show()

# Methods

## Generating Bounding Boxes

In [ ]:
boxes = dict()
for method in ['Manual','DeepForest','Detectree2','Baseline']:
    boxes[method] = dict()
    for density in ['Low','Medium','High']:
        boxes[method][density] = dict()

for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()

    # Collect hand-drawn bounding boxes
    manual_boxes = img['box']

    # Generate automatic bounding boxes using DeepForest
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        deepforest_preds = deepforest_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
    # Filter with confidence threshold of 0.2
    if deepforest_preds is not None:
        deepforest_preds = deepforest_preds[deepforest_preds['score'] >= 0.2]
        deepforest_scores = deepforest_preds['score'].to_numpy()
        deepforest_boxes = deepforest_preds[['xmin','ymin','xmax','ymax']].to_numpy()
    else:
        deepforest_scores = np.empty((0))
        deepforest_boxes = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
    detectree_preds = detectree_model(bgr_img)
    if detectree_preds is not None:
        detectree_boxes = detectree_preds['instances'].pred_boxes.tensor.cpu().numpy()
        detectree_scores = detectree_preds['instances'].scores.cpu().numpy()
        detectree_classes = detectree_preds['instances'].pred_classes.cpu().numpy()
        baseline_masks = detectree_preds['instances'].pred_masks.cpu().numpy()
        # Filter with confidence threshold of 0.2
        detectree_idx = detectree_scores >= 0.2
        detectree_boxes = detectree_boxes[detectree_idx]
        detectree_scores = detectree_scores[detectree_idx]
        detectree_classes = detectree_classes[detectree_idx]
        baseline_masks = baseline_masks[detectree_idx]
    else:
        detectree_boxes = np.empty((0,4))
        detectree_scores = np.empty((0))
        detectree_classes = np.empty((0))
        baseline_masks = np.empty(0,400,400)

    # Create detections objects for boxes
    manual_detections = sv.Detections(xyxy=manual_boxes, confidence=np.ones(len(manual_boxes)),
                                      class_id=np.zeros(len(manual_boxes), dtype='int'))
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, confidence=deepforest_scores,
                                          class_id=np.zeros(len(deepforest_boxes), dtype='int'))
    detectree_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores,
                                         class_id=detectree_classes)
    baseline_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores,
                                             class_id=detectree_classes)

    # Apply custom non-max suppression to automatic boxes
    deepforest_idx = custom_nms(deepforest_detections, threshold=0.5)
    detectree_idx = custom_nms(detectree_detections, threshold=0.5)
    deepforest_detections = deepforest_detections[deepforest_idx]
    detectree_detections = detectree_detections[detectree_idx]
    baseline_detections = baseline_detections[detectree_idx]
    baseline_masks = baseline_masks[detectree_idx]

    # Store predicted bounding boxes
    boxes['Manual'][density][filename] = {'Boxes':manual_detections}
    boxes['DeepForest'][density][filename] = {'Boxes':deepforest_detections}
    boxes['Detectree2'][density][filename] = {'Boxes':detectree_detections}
    boxes['Baseline'][density][filename] = {'Boxes':baseline_detections, 'Masks':baseline_masks}

Visualize DeepForest bounding boxes.

In [ ]:
box_annotator = sv.BoxAnnotator(thickness=2, color=sv.Color.RED)

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4))
for i, (filename, density) in enumerate([('2018_SJER_3_254000_4108000_image_700', 'Low'),
                                            ('WREF_072_2019', 'Medium'), ('DELA_047_2019', 'High')]):
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    deepforest_boxes = boxes['DeepForest'][density][filename]['Boxes'].xyxy
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, class_id=np.arange(len(deepforest_boxes)))

    box_img = box_annotator.annotate(bgr_img.copy(), detections=deepforest_detections)[:,:,::-1]
    axs[i].imshow(box_img)
    axs[i].set_frame_on(False)
    axs[i].get_xaxis().set_ticks([])
    axs[i].get_yaxis().set_ticks([])

plt.tight_layout()
#plt.savefig('Bounding Boxes.jpg')
plt.show()

## Incorporating LiDAR Data

The cell above in **Generating Bounding Boxes** must be run first.

In [ ]:
# Set number of positive and negative LiDAR points to sample. Number used in paper is 10 positive and 10 negative points.
pos = 10
neg = 10

lidar = dict()
for method in ['Manual','DeepForest','Detectree2','Baseline']:
    lidar[method] = dict()
    for density in ['Low','Medium','High']:
        lidar[method][density] = dict()

for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()

    # Retrieve predicted bounding boxes, scores, and classes
    manual_boxes = boxes['Manual'][density][filename]['Boxes'].xyxy
    deepforest_boxes = boxes['DeepForest'][density][filename]['Boxes'].xyxy
    detectree_boxes = boxes['Detectree2'][density][filename]['Boxes'].xyxy
    baseline_boxes = boxes['Baseline'][density][filename]['Boxes'].xyxy

    deepforest_scores = boxes['DeepForest'][density][filename]['Boxes'].confidence
    detectree_scores = boxes['Detectree2'][density][filename]['Boxes'].confidence
    baseline_scores = boxes['Baseline'][density][filename]['Boxes'].confidence

    detectree_classes = boxes['Detectree2'][density][filename]['Boxes'].class_id
    baseline_classes = boxes['Baseline'][density][filename]['Boxes'].confidence

    baseline_masks = boxes['Baseline'][density][filename]['Masks']

    # Use boxes to label LiDAR points
    manual_coords, manual_labels = rasterize_lidar(las=img['lidar'], tif=img['tif'], boxes=manual_boxes)
    deepforest_coords, deepforest_labels = rasterize_lidar(las=img['lidar'], tif=img['tif'], boxes=deepforest_boxes)
    detectree_coords, detectree_labels = rasterize_lidar(las=img['lidar'], tif=img['tif'], boxes=detectree_boxes)

    # Use labeled LiDAR points to filter boxes
    idx = lidar_filter(deepforest_boxes, deepforest_coords, deepforest_labels, threshold=0.2)
    deepforest_boxes, deepforest_labels = deepforest_boxes[idx], deepforest_labels[idx]
    deepforest_scores = deepforest_scores[idx]

    idx = lidar_filter(detectree_boxes, detectree_coords, detectree_labels, threshold=0.2)
    detectree_boxes, detectree_labels = detectree_boxes[idx], detectree_labels[idx],
    detectree_scores, detectree_classes = detectree_scores[idx], detectree_classes[idx]

    baseline_boxes, baseline_masks = baseline_boxes[idx], baseline_masks[idx]
    baseline_scores, baseline_classes = baseline_scores[idx], baseline_classes[idx]

    # Create detections objects
    manual_detections = sv.Detections(xyxy=manual_boxes, confidence=np.ones(len(manual_boxes)),
                                      class_id=np.zeros(len(manual_boxes), dtype='int'))
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, confidence=deepforest_scores,
                                          class_id=np.zeros(len(deepforest_boxes), dtype='int'))
    detectree_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores,
                                         class_id=detectree_classes)
    baseline_detections = sv.Detections(xyxy=baseline_boxes, confidence=baseline_scores,
                                        class_id=baseline_classes)

    # Sample LiDAR points for SAM-2. Parameters used in paper are distance_weight=True, neg_sample_spread=1, seed=1 (set above).
    manual_coords, manual_labels = sample_points(manual_coords, manual_labels, pos_samples=pos, neg_samples=neg,
                                                 distance_weight=True, neg_sample_spread=1, seed=seed)
    if len(deepforest_boxes) > 0:
        deepforest_coords, deepforest_labels = sample_points(deepforest_coords, deepforest_labels, pos_samples=pos, neg_samples=neg,
                                                             distance_weight=True, neg_sample_spread=1, seed=seed)
    else:
        deepforest_coords, deepforest_labels = None, None

    if len(detectree_boxes) > 0:
        detectree_coords, detectree_labels = sample_points(detectree_coords, detectree_labels, pos_samples=pos, neg_samples=neg,
                                                           distance_weight=True, neg_sample_spread=1, seed=seed)
    else:
        detectree_coords, detectree_labels = None, None

    lidar['Manual'][density][filename] = {'Boxes':manual_detections, 'LiDAR':(manual_coords, manual_labels)}
    lidar['DeepForest'][density][filename] = {'Boxes':deepforest_detections, 'LiDAR':(deepforest_coords, deepforest_labels)}
    lidar['Detectree2'][density][filename] = {'Boxes':detectree_detections, 'LiDAR':(detectree_coords, detectree_labels)}
    lidar['Baseline'][density][filename] = {'Boxes':baseline_detections, 'Masks':baseline_masks}

Visualize sampled LiDAR points with DeepForest boxes.

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4))
for i, (filename, density) in enumerate([('2018_SJER_3_254000_4108000_image_700', 'Low'),
                                            ('WREF_072_2019', 'Medium'), ('DELA_047_2019', 'High')]):
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    deepforest_boxes = lidar['DeepForest'][density][filename]['Boxes'].xyxy
    lidar_coords, lidar_labels = lidar['DeepForest'][density][filename]['LiDAR']
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, class_id=np.arange(len(deepforest_boxes)))

    axs[i] = show_as_points(rgb_img, deepforest_detections, lidar_coords[:5], lidar_labels[:5], ax=axs[i],
                            show_positive=True, show_negative=True, show_boxes=True, marker_size=25)
    axs[i].set_frame_on(False)
    axs[i].get_xaxis().set_ticks([])
    axs[i].get_yaxis().set_ticks([])

plt.tight_layout()
#plt.savefig('LiDAR Points.jpg')
plt.show()

## Prompting SAM

The cells above, **Generating Bounding Boxes** and **Incorporating LiDAR Data** must be run first. If running on Google Colab's T4 GPU with High RAM, this should take about 4 minutes; if running on Google Colab's CPU, it may take over 30 minutes.

In [ ]:
for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    sam_predictor.set_image(rgb_img)

    # Prompt SAM with Bounding Boxes
    manual_boxes = boxes['Manual'][density][filename]['Boxes']
    deepforest_boxes = boxes['DeepForest'][density][filename]['Boxes']
    detectree_boxes = boxes['Detectree2'][density][filename]['Boxes']

    manual_masks, manual_scores = segment_boxes(sam_predictor, manual_boxes.xyxy)
    if len(deepforest_boxes) > 0:
        deepforest_masks, deepforest_scores = segment_boxes(sam_predictor, deepforest_boxes.xyxy)
    else:
        deepforest_masks, deepforest_scores = np.empty((0,400,400)), np.empty((0))
    if len(detectree_boxes) > 0:
        detectree_masks, detectree_scores = segment_boxes(sam_predictor, detectree_boxes.xyxy)
    else:
        detectree_masks, detectree_scores = np.empty((0,400,400)), np.empty((0))

    # Filter masks with confidence threshold of 0.2
    idx = manual_scores >= 0.2
    boxes['Manual'][density][filename]['Boxes'] = manual_boxes[idx]
    boxes['Manual'][density][filename]['Masks'] = manual_masks[idx]

    idx = deepforest_scores >= 0.2
    boxes['DeepForest'][density][filename]['Boxes'] = deepforest_boxes[idx]
    boxes['DeepForest'][density][filename]['Masks'] = deepforest_masks[idx]

    idx = detectree_scores >= 0.2
    boxes['Detectree2'][density][filename]['Boxes'] = detectree_boxes[idx]
    boxes['Detectree2'][density][filename]['Masks'] = detectree_masks[idx]

    # Prompt SAM with Bounding Boxes and LiDAR
    manual_boxes = lidar['Manual'][density][filename]['Boxes']
    deepforest_boxes = lidar['DeepForest'][density][filename]['Boxes']
    detectree_boxes = lidar['Detectree2'][density][filename]['Boxes']

    manual_coords, manual_labels = lidar['Manual'][density][filename]['LiDAR']
    deepforest_coords, deepforest_labels = lidar['DeepForest'][density][filename]['LiDAR']
    detectree_coords, detectree_labels = lidar['Detectree2'][density][filename]['LiDAR']

  # Segment using SAM-2 on bounding boxes and LiDAR. In paper, iterative=True (default)
    manual_masks, manual_scores = segment_box_lidar(sam_predictor, manual_boxes.xyxy, manual_coords, manual_labels)
    if len(deepforest_boxes) > 0:
        deepforest_masks, deepforest_scores = segment_box_lidar(sam_predictor, deepforest_boxes.xyxy, deepforest_coords, deepforest_labels)
    else:
        deepforest_masks, deepforest_scores = np.empty((0,400,400)), np.empty((0))
    if len(detectree_boxes) > 0:
        detectree_masks, detectree_scores = segment_box_lidar(sam_predictor, detectree_boxes.xyxy, detectree_coords, detectree_labels)
    else:
        detectree_masks, detectree_scores = np.empty((0,400,400)), np.empty((0))

    # Filter masks with confidence threshold of 0.2
    idx = manual_scores >= 0.2
    lidar['Manual'][density][filename]['Boxes'] = manual_boxes[idx]
    lidar['Manual'][density][filename]['LiDAR'] = (manual_coords[idx], manual_labels[idx])
    lidar['Manual'][density][filename]['Masks'] = manual_masks[idx]

    idx = deepforest_scores >= 0.2
    lidar['DeepForest'][density][filename]['Boxes'] = deepforest_boxes[idx]
    lidar['DeepForest'][density][filename]['LiDAR'] = (deepforest_coords[idx], deepforest_labels[idx])
    lidar['DeepForest'][density][filename]['Masks'] = deepforest_masks[idx]

    idx = detectree_scores >= 0.2
    lidar['Detectree2'][density][filename]['Boxes'] = detectree_boxes[idx]
    lidar['Detectree2'][density][filename]['LiDAR'] = (detectree_coords[idx], detectree_labels[idx])
    lidar['Detectree2'][density][filename]['Masks'] = detectree_masks[idx]

Visualize ITC delineations with DeepForest + LiDAR.

In [ ]:
poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
mask_annotator = sv.MaskAnnotator(opacity=0.5, color=sv.Color.WHITE)

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4))
for i, (filename, density) in enumerate([('2018_SJER_3_254000_4108000_image_700', 'Low'),
                                            ('WREF_072_2019', 'Medium'), ('DELA_047_2019', 'High')]):
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    deepforest_boxes = lidar['DeepForest'][density][filename]['Boxes'].xyxy
    deepforest_masks = lidar['DeepForest'][density][filename]['Masks']
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, mask=deepforest_masks, class_id=np.arange(len(deepforest_boxes)))

    mask_img = mask_annotator.annotate(bgr_img.copy(), detections=deepforest_detections)
    poly_img = poly_annotator.annotate(mask_img, detections=deepforest_detections)[:,:,::-1]
    axs[i].imshow(poly_img)
    axs[i].set_frame_on(False)
    axs[i].get_xaxis().set_ticks([])
    axs[i].get_yaxis().set_ticks([])

plt.tight_layout()
#plt.savefig('ITC Delineations.jpg')
plt.show()

Visualize ITC delineations with all prompts on low density trees.

In [ ]:
filename, density = '2018_SJER_3_254000_4108000_image_700', 'Low'
img = dataset.get_image(filename)
rgb_img = img['rgb']
bgr_img = rgb_img[:,:,::-1].copy()
true_detections = sv.Detections(xyxy=img['box'], mask=img['mask'], class_id=np.arange(len(img['box'])))

poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
mask_annotator = sv.MaskAnnotator(opacity=0.5, color=sv.Color.WHITE)

true_img = mask_annotator.annotate(bgr_img.copy(), detections=true_detections)
true_img = poly_annotator.annotate(true_img, detections=true_detections)[:,:,::-1]

# Visualize
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(15,7.8))
for i, prompt in enumerate([boxes, lidar]):
    for j, method in enumerate(['Manual','DeepForest','Detectree2']):
        pred_boxes = prompt[method][density][filename]['Boxes'].xyxy
        pred_masks = prompt[method][density][filename]['Masks']
        pred_detections = sv.Detections(xyxy=pred_boxes, mask=pred_masks, class_id=np.arange(len(pred_boxes)))
        pred_img = mask_annotator.annotate(bgr_img.copy(), detections=pred_detections)
        pred_img = poly_annotator.annotate(pred_img, detections=pred_detections)[:,:,::-1]
        axs[i,j+1].imshow(pred_img)
        axs[i,j+1].set_frame_on(False)
        axs[i,j+1].get_xaxis().set_ticks([])
        axs[i,j+1].get_yaxis().set_ticks([])

axs[0,0].imshow(true_img)
axs[0,0].set_frame_on(False)
axs[0,0].get_xaxis().set_ticks([])
axs[0,0].get_yaxis().set_ticks([])
axs[1,0].set_frame_on(False)
axs[1,0].get_xaxis().set_ticks([])
axs[1,0].get_yaxis().set_ticks([])

axs[0,0].set_title('Ground Truth', fontsize=16, fontweight='bold')
axs[0,1].set_title('Manual', fontsize=16, fontweight='bold')
axs[0,2].set_title('DeepForest', fontsize=16, fontweight='bold')
axs[0,3].set_title('Detectree2', fontsize=16, fontweight='bold')
axs[0,1].set_ylabel('Boxes', fontsize=16, fontweight='bold')
axs[1,1].set_ylabel('Boxes + LiDAR', fontsize=16, fontweight='bold')

plt.tight_layout()
#plt.savefig('Low Density Delineations.jpg')
plt.show()

Visualize ITC delineations with all prompts on medium density trees.

In [ ]:
filename, density = 'WREF_072_2019', 'Medium'
img = dataset.get_image(filename)
rgb_img = img['rgb']
bgr_img = rgb_img[:,:,::-1].copy()
true_detections = sv.Detections(xyxy=img['box'], mask=img['mask'], class_id=np.arange(len(img['box'])))

poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
mask_annotator = sv.MaskAnnotator(opacity=0.5, color=sv.Color.WHITE)

true_img = mask_annotator.annotate(bgr_img.copy(), detections=true_detections)
true_img = poly_annotator.annotate(true_img, detections=true_detections)[:,:,::-1]

# Visualize
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(15,7.8))
for i, prompt in enumerate([boxes, lidar]):
    for j, method in enumerate(['Manual','DeepForest','Detectree2']):
        pred_boxes = prompt[method][density][filename]['Boxes'].xyxy
        pred_masks = prompt[method][density][filename]['Masks']
        pred_detections = sv.Detections(xyxy=pred_boxes, mask=pred_masks, class_id=np.arange(len(pred_boxes)))
        pred_img = mask_annotator.annotate(bgr_img.copy(), detections=pred_detections)
        pred_img = poly_annotator.annotate(pred_img, detections=pred_detections)[:,:,::-1]
        axs[i,j+1].imshow(pred_img)
        axs[i,j+1].set_frame_on(False)
        axs[i,j+1].get_xaxis().set_ticks([])
        axs[i,j+1].get_yaxis().set_ticks([])

axs[0,0].imshow(true_img)
axs[0,0].set_frame_on(False)
axs[0,0].get_xaxis().set_ticks([])
axs[0,0].get_yaxis().set_ticks([])
axs[1,0].set_frame_on(False)
axs[1,0].get_xaxis().set_ticks([])
axs[1,0].get_yaxis().set_ticks([])

axs[0,0].set_title('Ground Truth', fontsize=16, fontweight='bold')
axs[0,1].set_title('Manual', fontsize=16, fontweight='bold')
axs[0,2].set_title('DeepForest', fontsize=16, fontweight='bold')
axs[0,3].set_title('Detectree2', fontsize=16, fontweight='bold')
axs[0,1].set_ylabel('Boxes', fontsize=16, fontweight='bold')
axs[1,1].set_ylabel('Boxes + LiDAR', fontsize=16, fontweight='bold')

plt.tight_layout()
#plt.savefig('Medium Density Delineations.jpg')
plt.show()

Visualize ITC delineations with all prompts on high density trees.

In [ ]:
filename, density = 'DELA_047_2019', 'High'
img = dataset.get_image(filename)
rgb_img = img['rgb']
bgr_img = rgb_img[:,:,::-1].copy()
true_detections = sv.Detections(xyxy=img['box'], mask=img['mask'], class_id=np.arange(len(img['box'])))

poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
mask_annotator = sv.MaskAnnotator(opacity=0.5, color=sv.Color.WHITE)

true_img = mask_annotator.annotate(bgr_img.copy(), detections=true_detections)
true_img = poly_annotator.annotate(true_img, detections=true_detections)[:,:,::-1]

# Visualize
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(15,7.8))
for i, prompt in enumerate([boxes, lidar]):
    for j, method in enumerate(['Manual','DeepForest','Detectree2']):
        pred_boxes = prompt[method][density][filename]['Boxes'].xyxy
        pred_masks = prompt[method][density][filename]['Masks']
        pred_detections = sv.Detections(xyxy=pred_boxes, mask=pred_masks, class_id=np.arange(len(pred_boxes)))
        pred_img = mask_annotator.annotate(bgr_img.copy(), detections=pred_detections)
        pred_img = poly_annotator.annotate(pred_img, detections=pred_detections)[:,:,::-1]
        axs[i,j+1].imshow(pred_img)
        axs[i,j+1].set_frame_on(False)
        axs[i,j+1].get_xaxis().set_ticks([])
        axs[i,j+1].get_yaxis().set_ticks([])

axs[0,0].imshow(true_img)
axs[0,0].set_frame_on(False)
axs[0,0].get_xaxis().set_ticks([])
axs[0,0].get_yaxis().set_ticks([])
axs[1,0].set_frame_on(False)
axs[1,0].get_xaxis().set_ticks([])
axs[1,0].get_yaxis().set_ticks([])

axs[0,0].set_title('Ground Truth', fontsize=16, fontweight='bold')
axs[0,1].set_title('Manual', fontsize=16, fontweight='bold')
axs[0,2].set_title('DeepForest', fontsize=16, fontweight='bold')
axs[0,3].set_title('Detectree2', fontsize=16, fontweight='bold')
axs[0,1].set_ylabel('Boxes', fontsize=16, fontweight='bold')
axs[1,1].set_ylabel('Boxes + LiDAR', fontsize=16, fontweight='bold')

plt.tight_layout()
#plt.savefig('High Density Delineations.jpg')
plt.show()

# Results

## Object-based and Pixel-based metrics

Average metrics across all plots

In [ ]:
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

prompts = [boxes['Manual'], lidar['Manual'], boxes['DeepForest'], lidar['DeepForest'],
           boxes['Detectree2'], lidar['Detectree2'], boxes['Baseline'], lidar['Baseline']]

columns = [('Object','Precision'), ('Object','Recall'),('Object','F1'),
           ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1'),
           ('Pixel','IoU')]

columns = pd.MultiIndex.from_tuples(columns)

metrics_all = pd.DataFrame(index=methods, columns=columns)

truths = []
for filename, density in test_imgs:
    img = dataset.get_image(filename)
    true_boxes = img['box']
    true_masks = img['mask']
    truths.append(sv.Detections(xyxy=true_boxes, mask=true_masks))

for method, prompt in zip(methods, prompts):
    preds = []
    for filename, density in test_imgs:
        pred_boxes = prompt[density][filename]['Boxes']
        pred_boxes.mask = prompt[density][filename]['Masks']
        preds.append(pred_boxes)
    metrics_all.loc[method] = per_tree_metrics(truths, preds)

display(metrics_all.style.format('{:.1%}'))

In [ ]:
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(12,8), sharey=True)
x = np.arange(4)
width = 0.4
yticks = np.arange(0.1, 1.1, 0.2)

for j, metric in enumerate(['Precision','Recall','F1']):
    box_objects = [metrics_all.loc[methods[i], ('Object',f'{metric}')] for i in range(0,len(methods),2)]
    lidar_objects = [metrics_all.loc[methods[i], ('Object',f'{metric}')] for i in range(1,len(methods),2)]
    box_pixel = [metrics_all.loc[methods[i], ('Pixel',f'{metric}')] for i in range(0,len(methods),2)]
    lidar_pixel = [metrics_all.loc[methods[i], ('Pixel',f'{metric}')] for i in range(1,len(methods),2)]
    object_metrics = list(zip(box_objects, lidar_objects))
    pixel_metrics = list(zip(box_pixel, lidar_pixel))

    for i in range(2):
        offset = width * i
        colors =['Bounding Box','LiDAR']
        obj_bars = axs[0,j].bar(x+offset, [metric[i] for metric in object_metrics], width=width, label=colors[i])
        pix_bars = axs[1,j].bar(x+offset, [metric[i] for metric in pixel_metrics], width=width, label=colors[i])
        axs[0,j].bar_label(obj_bars, fmt='{:.0%}',padding=3)
        axs[1,j].bar_label(pix_bars, fmt='{:.0%}',padding=3)

    for i in range(2):
        axs[i,j].set_xticks(x+width/2-0.01, ['Manual','DeepForest','Detectree2','Baseline'])
        axs[i,j].set_ylim(0,1.05)
        axs[i,j].set_yticks(yticks, [f'{n:.0%}' for n in yticks])

    axs[0,j].set_title(metric)
    axs[1,j].set_title(metric)
axs[0,0].set_ylabel('Object-based metrics')
axs[1,0].set_ylabel('Pixel-based metrics')
axs[0,0].legend()
fig.suptitle('Average Metrics Across All Plots')
plt.tight_layout()
#plt.savefig('Average Metrics Across All Plots.pdf', dpi=500)
plt.show()

## Effects of Bounding Box Algorithms

Compare bounding box vs delineation metrics

In [ ]:
box_methods = ['Manual Boxes', 'DeepForest Boxes', 'Detectree2 Boxes', 'Baseline']

box_prompts = [boxes['Manual'], boxes['DeepForest'], boxes['Detectree2'], boxes['Baseline']]

densities = ['Low','Medium', 'High']

columns = [('Bounding Box','Precision'), ('Bounding Box','Recall'),('Bounding Box','F1'),
           ('Delineation','Precision'), ('Delineation','Recall'), ('Delineation','F1')]
columns = pd.MultiIndex.from_tuples(columns)

metrics_box_mask = pd.DataFrame(index=box_methods, columns=columns)

truths = []
for filename, density in test_imgs:
    img = dataset.get_image(filename)
    true_boxes = img['box']
    true_masks = img['mask']
    truths.append(sv.Detections(xyxy=true_boxes, mask=true_masks))

for method, prompt in zip(box_methods, box_prompts):
    preds = []
    for filename, density in test_imgs:
        pred_boxes = prompt[density][filename]['Boxes']
        pred_boxes.mask = prompt[density][filename]['Masks']
        preds.append(pred_boxes)
    box_metrics = per_box_metrics(truths, preds)
    mask_metrics = per_tree_metrics(truths, preds)[:3]
    metrics_box_mask.loc[method] = box_metrics + mask_metrics

display(metrics_box_mask.style.format('{:.1%}'))

In [ ]:
box_methods = ['Manual Boxes', 'DeepForest Boxes', 'Detectree2 Boxes']

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4), sharey=True)
x = np.arange(3)
width = 0.4
yticks = np.arange(0.1, 1.1, 0.2)

for j, metric in enumerate(['Precision','Recall','F1']):
    box_objects = [metrics_box_mask.loc[method, ('Bounding Box',f'{metric}')] for method in box_methods]
    mask_objects = [metrics_box_mask.loc[method, ('Delineation',f'{metric}')] for method in box_methods]
    object_metrics = list(zip(box_objects, mask_objects))

    for i in range(2):
        offset = width * i
        colors =['Bounding Box','Delineation']
        obj_bars = axs[j].bar(x+offset, [metric[i] for metric in object_metrics], width=width, label=colors[i])
        axs[j].bar_label(obj_bars, fmt='{:.0%}',padding=3)

    axs[j].set_xticks(x+width/2-0.01, ['Manual','DeepForest','Detectree2'])
    axs[j].set_ylim(0,1.1)
    axs[j].set_yticks(yticks, [f'{n:.0%}' for n in yticks])

    axs[j].set_title(metric)
    axs[j].set_title(metric)

axs[0].set_ylabel('Object-based metrics')
axs[0].legend()
fig.suptitle('Bounding Box vs Delineation Metrics')
plt.tight_layout()
#plt.savefig('Bounding Box vs Delineation Metrics.pdf', dpi=500)
plt.show()

Compare bounding box prompt sizes

In [ ]:
box_methods = ['Manual Boxes', 'DeepForest Boxes', 'Detectree2 Boxes']

densities = ['Low', 'Medium', 'High']

manual_boxes, deepforest_boxes, detectree_boxes = [],[],[]

for filename, density in test_imgs:
    manual_boxes.append(boxes['Manual'][density][filename]['Boxes'].xyxy)
    deepforest_boxes.append(boxes['DeepForest'][density][filename]['Boxes'].xyxy)
    detectree_boxes.append(boxes['Detectree2'][density][filename]['Boxes'].xyxy)

manual_boxes = np.concatenate(manual_boxes)
deepforest_boxes = np.concatenate(deepforest_boxes)
detectree_boxes = np.concatenate(detectree_boxes)

manual_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in manual_boxes]
deepforest_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in deepforest_boxes]
detectree_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in detectree_boxes]

fig, axs = plt.subplots(figsize=(4,3.5))
axs.boxplot([manual_boxes, deepforest_boxes, detectree_boxes])
axs.set_xticks([1,2,3],['Manual', 'DeepForest', 'Detectree2'])
axs.set_ylabel('Bounding box size (pixels)')
axs.set_ylim(-1000,30000)
plt.tight_layout()
#plt.savefig('Bounding Box Size Box Plots (Clipped).pdf', dpi=500)
plt.show()

## Effects of LiDAR Data

Display Delineation Size with vs without LiDAR

In [ ]:
# This is close to the largest true segmentation
size_limit = 20000

fig, axs = plt.subplots(nrows=3, ncols=3, sharex=True, sharey=True, figsize=(7.5,7.5))

for j, method in enumerate(['Manual','DeepForest','Detectree2']):
    for filename, density in test_imgs:
        img = dataset.get_image(filename)
        true_masks = img['mask']
        box_masks = boxes[method][density][filename]['Masks']
        lidar_masks = lidar[method][density][filename]['Masks']

        true_sizes = np.array([mask.sum() for mask in true_masks])
        box_sizes = np.array([mask.sum() for mask in box_masks])
        lidar_sizes = np.array([mask.sum() for mask in lidar_masks])

        box_idx, lidar_idx = hungarian_matching(box_masks, lidar_masks, threshold=0.5)
        axs[0,j].scatter(box_sizes[box_idx], lidar_sizes[lidar_idx], s=10, c='tab:blue')

        true_idx, lidar_idx = hungarian_matching(true_masks, lidar_masks, threshold=0.5)
        axs[1,j].scatter(true_sizes[true_idx], lidar_sizes[lidar_idx], s=10, c='tab:blue')

        true_idx, box_idx = hungarian_matching(true_masks, box_masks, threshold=0.5)
        axs[2,j].scatter(true_sizes[true_idx], box_sizes[box_idx], s=10, c='tab:blue')

fig.suptitle('Delineation Size with vs without LiDAR')
axs[0,0].set_title('Manual Boxes')
axs[0,1].set_title('DeepForest Boxes')
axs[0,2].set_title('Detectree2 Boxes')

axs[0,0].set_ylabel('With LiDAR')
axs[1,0].set_ylabel('With LiDAR')
axs[2,0].set_ylabel('Without LiDAR')
for i in range(3):
    axs[0,i].set_xlabel('Without LiDAR')
    axs[1,i].set_xlabel('Ground Truth')
    axs[2,i].set_xlabel('Ground Truth')
    for j in range(3):
        axs[i,j].set_ylim(0,size_limit)
        axs[i,j].set_xlim(0,size_limit)
        axs[i,j].plot([0, 1], [0, 1], transform=axs[i,j].transAxes, linestyle='dashed')

plt.tight_layout()
#plt.savefig('Mask Sizes.pdf', dpi=500)
plt.show()

In [ ]:
lidar_methods = ['DeepForest + LiDAR', 'Detectree2 + LiDAR']

columns = [('Object','Precision'), ('Object','Recall'),('Object','F1'),
           ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1'),
           ('Pixel','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

filter_metrics = pd.DataFrame(index=lidar_methods, columns=columns)
prompt_metrics = pd.DataFrame(index=lidar_methods, columns=columns)

pos = 10
neg = 10

lidar_as_filter = deepcopy(boxes)
lidar_as_prompt = deepcopy(lidar)

truths = []
for filename, density in test_imgs:
    img = dataset.get_image(filename)
    true_boxes = img['box']
    true_masks = img['mask']
    truths.append(sv.Detections(xyxy=true_boxes, mask=true_masks))

for prompt in ['DeepForest','Detectree2']:
    filter_preds = []
    prompt_preds = []
    for filename, density in test_imgs:
        img = dataset.get_image(filename)
        rgb_img = img['rgb']

        # Get previously predicted box and lidar prompts
        pred_boxes = boxes[prompt][density][filename]['Boxes']
        pred_masks = boxes[prompt][density][filename]['Masks']
        pred_lidar = lidar[prompt][density][filename]['Boxes']

        # Match prompts between bounding box and lidar sets. Record which boxes were
        # filtered out by lidar.
        lidar_idx, box_idx = hungarian_matching(pred_lidar.xyxy, pred_boxes.xyxy)
        missing_idx = [i for i in range(len(pred_boxes.xyxy)) if i not in box_idx]

        if len(missing_idx) > 0:
            # For lidar_as_filter, filter out box prompt predictions based on lidar filter.
            lidar_as_filter[prompt][density][filename]['Boxes'] = pred_boxes[box_idx]
            lidar_as_filter[prompt][density][filename]['Masks'] = pred_masks[box_idx]

            # For lidar_as_prompt, add back in lidar prompt predictions that had been filtered out.
            missing_boxes = pred_boxes.xyxy[missing_idx]
            missing_coords, missing_labels = rasterize_lidar(las=img['lidar'], tif=img['tif'], boxes=missing_boxes)
            missing_coords, missing_labels = sample_points(missing_coords, missing_labels, pos_samples=pos, neg_samples=neg,
                                                       distance_weight=True, neg_sample_spread=1, seed=seed)
            sam_predictor.set_image(rgb_img)
            missing_masks, missing_scores = segment_box_lidar(sam_predictor, missing_boxes, missing_coords, missing_labels)

            idx = missing_scores >= 0.2
            missing_boxes, missing_masks, missing_scores = missing_boxes[idx], missing_masks[idx], missing_scores[idx]
            missing_coords, missing_labels = missing_coords[idx], missing_labels[idx]
            missing_detections = sv.Detections(xyxy=missing_boxes, mask=missing_masks, confidence=missing_scores,
                                               class_id=np.zeros(len(missing_boxes),dtype='int'))

            old_boxes = lidar[prompt][density][filename]['Boxes']
            old_coords, old_labels = lidar[prompt][density][filename]['LiDAR']
            old_masks = lidar[prompt][density][filename]['Masks']

            lidar_as_prompt[prompt][density][filename]['Boxes'] = sv.Detections.merge([old_boxes, missing_detections])
            lidar_as_prompt[prompt][density][filename]['LiDAR'] = (np.concatenate([old_coords, missing_coords]), np.concatenate([old_labels, missing_labels]))
            lidar_as_prompt[prompt][density][filename]['Masks'] = np.concatenate([old_masks, missing_masks])

        filter_preds.append(lidar_as_filter[prompt][density][filename]['Boxes'])
        prompt_preds.append(lidar_as_prompt[prompt][density][filename]['Boxes'])

    filter_metrics.loc[f'{prompt} + LiDAR'] = per_tree_metrics(truths, filter_preds)
    prompt_metrics.loc[f'{prompt} + LiDAR'] = per_tree_metrics(truths, prompt_preds)

print('LiDAR as Filter')
display(filter_metrics.style.format('{:.1%}'))
print()
print()
print('LiDAR as Prompt')
display(prompt_metrics.style.format('{:.1%}'))
print()
print()
print('LiDAR as Filter and Prompt')
display(metrics_all.loc[lidar_methods].style.format('{:.1%}'))

In [ ]:
lidar_methods = ['DeepForest + LiDAR', 'Detectree2 + LiDAR']

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12,4), sharey=True)
x = np.arange(2)
width = 0.3
yticks = np.arange(0.1, 1.1, 0.2)

for j, metric in enumerate(['Precision','Recall','F1']):
    filter_objects = [filter_metrics.loc[method, ('Object',f'{metric}')] for method in lidar_methods]
    prompt_objects = [prompt_metrics.loc[method, ('Object',f'{metric}')] for method in lidar_methods]
    both_objects = [metrics_all.loc[method, ('Object',f'{metric}')] for method in lidar_methods]
    object_metrics = list(zip(filter_objects, prompt_objects, both_objects))

    for i in range(3):
        offset = width * i
        colors =['Filter','Prompt','Both']
        obj_bars = axs[j].bar(x+offset, [metric[i] for metric in object_metrics], width=width, label=colors[i])
        axs[j].bar_label(obj_bars, fmt='{:.0%}',padding=3)

    axs[j].set_xticks(x+width/2-0.01, ['DeepForest','Detectree2'])
    axs[j].set_ylim(0,1.1)
    axs[j].set_yticks(yticks, [f'{n:.0%}' for n in yticks])

    axs[j].set_title(metric)
    axs[j].set_title(metric)

axs[0].set_ylabel('Object-based metrics')
axs[0].legend()
fig.suptitle('LiDAR as Filter vs Prompt')
plt.tight_layout()
#plt.savefig('LiDAR as Filter vs Prompt.pdf', dpi=500)
plt.show()

## Effects of Tree Density

Average metrics across densities

In [ ]:
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

prompts = [boxes['Manual'], lidar['Manual'], boxes['DeepForest'], lidar['DeepForest'],
           boxes['Detectree2'], lidar['Detectree2'], boxes['Baseline'], lidar['Baseline']]

densities = ['Low','Medium','High']

columns = [('Object','Precision'), ('Object','Recall'),('Object','F1'),
           ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1'),
           ('Pixel','IoU')]

columns = pd.MultiIndex.from_tuples(columns)

index = [(density, method) for density in densities for method in methods]
index = pd.MultiIndex.from_tuples(index, names=['Density', 'Method'])
metrics_density = pd.DataFrame(index=index, columns=columns)

for density in densities:
    for method, prompt in zip(methods, prompts):
        truths = []
        preds = []
        for filename, d in test_imgs:
            if d == density:
                img = dataset.get_image(filename)
                true_boxes = img['box']
                true_masks = img['mask']
                truths.append(sv.Detections(xyxy=true_boxes, mask=true_masks))

                pred_boxes = prompt[density][filename]['Boxes']
                pred_boxes.mask = prompt[density][filename]['Masks']
                preds.append(pred_boxes)
        metrics_density.loc[density, method] = per_tree_metrics(truths, preds)

display(metrics_density.style.format('{:.1%}'))

In [ ]:
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

densities = ['Low','Medium','High']

fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(8,12), sharey=True)
x = np.arange(4)
width = 0.15
yticks = np.arange(0.1, 1.1, 0.2)
colors = ['tab:blue', 'tab:orange']
patterns = ['/','','\\']

for i, metric in enumerate(['Precision','Recall','F1']):
    object_metrics = [metrics_density.loc[(density, method), ('Object',f'{metric}')] for method in methods for density in densities]
    object_metrics = [object_metrics[n:n+7] for n in range(0,24,6)]
    axs[i].set_title(metric)
    axs[i].set_ylabel('Object-based metrics')

    for j, prompt in enumerate(['Bounding Box','LiDAR']):
        for k in range(3):
            m = (j * 3) + k
            offset = width * m
            obj_bars = axs[i].bar(x+offset, [metric[m] for metric in object_metrics], width=width, color=colors[j], hatch=patterns[k], edgecolor='black')

    for k in range(4):
        axs[i].set_xticks(x+3*width-0.075, ['Manual','DeepForest','Detectree2','Baseline'])
        axs[i].set_ylim(0,1.05)
        axs[i].set_yticks(yticks, [f'{n:.0%}' for n in yticks])

box_patch = mpatches.Patch(color='tab:blue', label='Bounding Box')
lidar_patch = mpatches.Patch(color='tab:orange', label='LiDAR')
low_patch = mpatches.Patch(edgecolor='black', facecolor='white', hatch='///', label='Low Density')
medium_patch = mpatches.Patch(edgecolor='black', facecolor='white', label='Medium Density')
high_patch = mpatches.Patch(edgecolor='black', facecolor='white', hatch='\\\\\\', label='High Density')
axs[0].legend(handles=[low_patch, medium_patch, high_patch, box_patch, lidar_patch], ncol=2)
fig.suptitle(f'Average Object-based Metrics Across Tree Densities')
plt.tight_layout()
#plt.savefig(f'Average Object-based Metrics Across Tree Densities.pdf', dpi=500)
plt.show()

In [ ]:
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

densities = ['Low','Medium','High']

fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(8,12), sharey=True)
x = np.arange(4)
width = 0.15
yticks = np.arange(0.1, 1.1, 0.2)
colors = ['tab:blue', 'tab:orange']
patterns = ['/','','\\']

for i, metric in enumerate(['Precision','Recall','F1']):
    object_metrics = [metrics_density.loc[(density, method), ('Pixel',f'{metric}')] for method in methods for density in densities]
    object_metrics = [object_metrics[n:n+7] for n in range(0,24,6)]
    axs[i].set_title(metric)
    axs[i].set_ylabel('Pixel-based metrics')

    for j, prompt in enumerate(['Bounding Box','LiDAR']):
        for k in range(3):
            m = (j * 3) + k
            offset = width * m
            obj_bars = axs[i].bar(x+offset, [metric[m] for metric in object_metrics], width=width, color=colors[j], hatch=patterns[k], edgecolor='black')

    for k in range(4):
        axs[i].set_xticks(x+3*width-0.075, ['Manual','DeepForest','Detectree2','Baseline'])
        axs[i].set_ylim(0,1.05)
        axs[i].set_yticks(yticks, [f'{n:.0%}' for n in yticks])

box_patch = mpatches.Patch(color='tab:blue', label='Bounding Box')
lidar_patch = mpatches.Patch(color='tab:orange', label='LiDAR')
low_patch = mpatches.Patch(edgecolor='black', facecolor='white', hatch='///', label='Low Density')
medium_patch = mpatches.Patch(edgecolor='black', facecolor='white', label='Medium Density')
high_patch = mpatches.Patch(edgecolor='black', facecolor='white', hatch='\\\\\\', label='High Density')
axs[1].legend(handles=[low_patch, medium_patch, high_patch, box_patch, lidar_patch], ncol=2)
fig.suptitle(f'Average Pixel-based Metrics Across Tree Densities')
plt.tight_layout()
#plt.savefig(f'Average Pixel-based Metrics Across Tree Densities.pdf', dpi=500)
plt.show()

## Metrics Per Plot

In [ ]:
# Metrics by plot

methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

prompts = [boxes['Manual'], lidar['Manual'], boxes['DeepForest'], lidar['DeepForest'],
           boxes['Detectree2'], lidar['Detectree2'], boxes['Baseline'], lidar['Baseline']]

densities = ['Low','Medium','High']

columns = [('Object','Precision'), ('Object','Recall'),('Object','F1'),
           ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1'),
           ('Pixel','IoU')]

columns = pd.MultiIndex.from_tuples(columns)

index = [(density, method, filename) for method in methods for filename, density in test_imgs]
index = pd.MultiIndex.from_tuples(index, names=['Density', 'Method', 'Filename'])
metrics_plots = pd.DataFrame(index=index, columns=columns)

for filename, density in test_imgs:
    img = dataset.get_image(filename)
    true_boxes = img['box']
    true_masks = img['mask']
    truths = [sv.Detections(xyxy=true_boxes, mask=true_masks)]
    for method, prompt in zip(methods, prompts):
        pred_boxes = prompt[density][filename]['Boxes']
        pred_boxes.mask = prompt[density][filename]['Masks']
        preds = [pred_boxes]
        metrics_plots.loc[density, method, filename] = per_tree_metrics(truths, preds)

display(metrics_plots.style.format('{:.1%}'))